# Alert Governance Memo

This notebook is the **first governance bridge** in the curriculum: it turns tiny private-banking and digital-banking baseline outputs into alert interpretation, threshold rationale, investigation-capacity discussion, and governance-ready language for business, risk, or compliance stakeholders.

It stays the first bridge while the deeper layers extend around it: v0.5 case narratives add investigator reasoning and scam outcomes, v0.6 graph evidence adds network signals (`mule_ring`, `circular_funds_movement`), and the v0.7 `07_interpretability_model_risk` module adds the full model-documentation template, monitoring checklist, and governance memo. This notebook cross-references that module (see the final cell) so the two governance paths stay bidirectionally linked.

In [ ]:
import pandas as pd

from banking_fraud_lab import (
    build_learner_facing_views,
    evaluate_alert_scores,
    generate_digital_scam_to_mule_world,
    generate_private_banking_transaction_fraud_world,
    load_tables_to_sqlite,
)
from banking_fraud_lab.generators.digital_banking import DIGITAL_SCAM_TO_MULE_ACTIVITY_TYPE
from banking_fraud_lab.generators.private_banking import PRIVATE_BANKING_ACTIVITY_TYPE

## Baseline Outputs

The governance view consumes tiny-data equivalents of the two v0.1 baseline notebooks. Each baseline produces alert scores from learner-facing fields, then the shared evaluation utility reports alert-aware metrics.

In [ ]:
def private_banking_baseline() -> dict[str, object]:
    tables = generate_private_banking_transaction_fraud_world(
        seed=42,
        scenario_prevalence=0.2,
    )
    connection = load_tables_to_sqlite(build_learner_facing_views(tables), ":memory:")
    try:
        features = pd.read_sql_query(
            """
            WITH role_counts AS (
                SELECT banking_relationship_id, COUNT(*) AS relationship_role_count
                FROM partner_roles
                GROUP BY banking_relationship_id
            )
            SELECT
                c.case_id,
                c.alert_id,
                al.alert_type,
                CAST(t.amount_chf AS REAL) AS amount_chf,
                CAST(a.balance_chf AS REAL) AS balance_chf,
                rc.relationship_role_count,
                CAST(co.confirmed_fraud AS INTEGER) AS confirmed_fraud,
                CAST(co.loss_amount_chf AS REAL) AS loss_amount_chf
            FROM cases AS c
            JOIN alerts AS al ON c.alert_id = al.alert_id
            JOIN transactions AS t ON c.transaction_id = t.transaction_id
            JOIN accounts AS a ON c.account_id = a.account_id
            JOIN role_counts AS rc ON c.banking_relationship_id = rc.banking_relationship_id
            JOIN case_outcomes AS co ON c.case_id = co.case_id
            WHERE al.institution_name = 'Alpine Crest Private Bank'
            ORDER BY c.opened_at
            """,
            connection,
        )
    finally:
        connection.close()

    features["amount_to_balance"] = features["amount_chf"] / features["balance_chf"].clip(lower=1.0)
    features["score"] = (
        0.50 * (features["amount_to_balance"] * 3).clip(upper=1.0)
        + 0.35 * features["alert_type"].eq(PRIVATE_BANKING_ACTIVITY_TYPE).astype(float)
        + 0.15 * (features["relationship_role_count"] >= 2).astype(float)
    ).clip(upper=1.0)
    report = evaluate_alert_scores(
        features[["case_id", "alert_id"]],
        features[["case_id", "confirmed_fraud", "loss_amount_chf"]],
        features[["alert_id", "score"]],
        thresholds=(0.35, 0.55, 0.75),
        alert_capacity=2,
        investigation_cost_chf=100.0,
        false_positive_cost_chf=25.0,
    )
    return {
        "track": "Private-banking fraud detection",
        "institution": "Alpine Crest Private Bank",
        "features": features,
        "report": report,
    }


def digital_banking_baseline() -> dict[str, object]:
    tables = generate_digital_scam_to_mule_world(seed=42, scenario_prevalence=1.0)
    connection = load_tables_to_sqlite(build_learner_facing_views(tables), ":memory:")
    try:
        features = pd.read_sql_query(
            """
            WITH device_user_counts AS (
                SELECT device_fingerprint_hash, COUNT(DISTINCT user_id) AS device_user_count
                FROM sessions
                GROUP BY device_fingerprint_hash
            )
            SELECT
                c.case_id,
                c.alert_id,
                al.alert_type,
                JULIANDAY(t.booked_at) - JULIANDAY(a.opened_at) AS account_age_days,
                s.asn_risk_score,
                s.is_vpn_or_proxy,
                pb.beneficiary_change_event,
                duc.device_user_count,
                CAST(co.confirmed_fraud AS INTEGER) AS confirmed_fraud,
                CAST(co.loss_amount_chf AS REAL) AS loss_amount_chf
            FROM cases AS c
            JOIN alerts AS al ON c.alert_id = al.alert_id
            JOIN transactions AS t ON c.transaction_id = t.transaction_id
            JOIN accounts AS a ON c.account_id = a.account_id
            LEFT JOIN sessions AS s ON c.session_id = s.session_id
            LEFT JOIN payment_beneficiaries AS pb ON c.payment_beneficiary_id = pb.payment_beneficiary_id
            LEFT JOIN device_user_counts AS duc ON s.device_fingerprint_hash = duc.device_fingerprint_hash
            JOIN case_outcomes AS co ON c.case_id = co.case_id
            WHERE al.institution_name = 'NovaBank Digital'
            ORDER BY c.opened_at
            """,
            connection,
        )
    finally:
        connection.close()

    features["account_age_days"] = features["account_age_days"].fillna(999.0)
    features["device_user_count"] = features["device_user_count"].fillna(1)
    features["asn_risk_score"] = features["asn_risk_score"].fillna(0)
    features["is_vpn_or_proxy"] = features["is_vpn_or_proxy"].fillna(0).astype(int)
    features["score"] = (
        0.30 * (features["account_age_days"] <= 7).astype(float)
        + 0.25 * features["beneficiary_change_event"].eq("new_beneficiary_added").astype(float)
        + 0.20 * (features["device_user_count"] > 1).astype(float)
        + 0.15 * ((features["asn_risk_score"] >= 80) | (features["is_vpn_or_proxy"] == 1)).astype(float)
        + 0.10 * features["alert_type"].eq(DIGITAL_SCAM_TO_MULE_ACTIVITY_TYPE).astype(float)
    ).clip(upper=1.0)
    report = evaluate_alert_scores(
        features[["case_id", "alert_id"]],
        features[["case_id", "confirmed_fraud", "loss_amount_chf"]],
        features[["alert_id", "score"]],
        thresholds=(0.35, 0.6, 0.8),
        alert_capacity=2,
        investigation_cost_chf=35.0,
        false_positive_cost_chf=10.0,
    )
    return {
        "track": "Digital-banking fraud detection",
        "institution": "NovaBank Digital",
        "features": features,
        "report": report,
    }


baselines = [private_banking_baseline(), digital_banking_baseline()]
assert all(baseline["report"]["population"]["case_count"] > 0 for baseline in baselines)
[baseline["institution"] for baseline in baselines]

## Threshold Summary

Alert governance needs the operational view: alert volume, precision and recall tradeoffs, threshold rationale, and whether expected alerts fit available investigation capacity.

In [ ]:
rows = []
for baseline in baselines:
    report = baseline["report"]
    best = report["lowest_cost_summary"]
    rows.append(
        {
            "track": baseline["track"],
            "institution": baseline["institution"],
            "case_count": report["population"]["case_count"],
            "confirmed_fraud_count": report["population"]["confirmed_fraud_count"],
            "pr_auc": report["pr_auc"],
            "selected_threshold": report["lowest_cost_threshold"],
            "precision": best["precision"],
            "recall": best["recall"],
            "alert_volume": best["alert_volume"],
            "alert_capacity": best["alert_capacity"],
            "over_capacity_alerts": best["over_capacity_alerts"],
            "total_cost_chf": best["total_cost_chf"],
        }
    )

governance_summary = pd.DataFrame(rows)
assert governance_summary["alert_volume"].le(governance_summary["alert_capacity"]).all()
governance_summary

In [ ]:
threshold_detail = pd.concat(
    [
        pd.DataFrame(baseline["report"]["threshold_summaries"]).assign(
            track=baseline["track"],
            institution=baseline["institution"],
            pr_auc=baseline["report"]["pr_auc"],
        )
        for baseline in baselines
    ],
    ignore_index=True,
)
threshold_detail[
    [
        "track",
        "threshold",
        "pr_auc",
        "precision",
        "recall",
        "alert_volume",
        "alert_capacity",
        "over_capacity_alerts",
        "total_cost_chf",
    ]
]

## Cross-Version Evidence

Alert governance draws on evidence from every version, not just the baseline threshold summary. Keeping the signal types side by side reminds reviewers that a threshold is one view among several.

| Layer | Version | What it adds to governance |
|---|---|---|
| Rule / threshold | v0.1 (this notebook) | Deterministic alert volume, capacity, precision/recall, cost. |
| Case narrative | v0.5 | Investigator reasoning and confirmed scam outcome per case. |
| Graph evidence | v0.6 | Network signals: `mule_ring` (digital) and `circular_funds_movement` (private banking). |
| Model-risk templates | v0.7 | Model documentation, monitoring checklist, and the full governance memo (`07_interpretability_model_risk`). |

### Why depth matters here

The threshold summary below is the operational view; the v0.5 case layer explains *why* an alert was confirmed or dismissed, the v0.6 graph layer shows when an alert sits inside a network pattern, and the v0.7 templates turn all of that into a reviewable artifact. A model should not be judged by headline accuracy — the deeper layers exist precisely to keep that limitation visible.

## Governance Memo Template

The memo below is generated from the tiny baseline outputs. Learners can adapt the structure when discussing threshold choices and limitations with stakeholders.

In [ ]:
def memo_section(row: pd.Series) -> str:
    capacity_note = (
        "within current investigation capacity"
        if row["over_capacity_alerts"] == 0
        else f"{int(row['over_capacity_alerts'])} alerts over current capacity"
    )
    return (
        f"### {row['institution']} - {row['track']}\n"
        f"Recommended threshold: {row['selected_threshold']:.2f}. "
        f"At this threshold, expected alert volume is {int(row['alert_volume'])} "
        f"against capacity of {int(row['alert_capacity'])}, which is {capacity_note}.\n\n"
        f"Precision is {row['precision']:.2f} and recall is {row['recall']:.2f}; "
        f"PR-AUC is {row['pr_auc']:.2f}. The threshold rationale is to balance "
        "fraud detection against investigation workload and estimated review cost.\n\n"
        "Limitations: results use deterministic synthetic tiny data and generated case "
        "outcomes. They support governance discussion, not production approval. Do not "
        "use headline accuracy for this decision.\n"
    )


memo = "\n".join(
    [
        "# Governance Memo Draft",
        "Audience: business, risk, and compliance stakeholders.",
        "Purpose: document alert threshold rationale, capacity impact, and model-output limitations.",
        *[memo_section(row) for _, row in governance_summary.iterrows()],
        "## Follow-Up Questions",
        "- Is investigation capacity stable for the next review period?",
        "- Which false-positive costs and missed-fraud costs should be challenged by human reviewers?",
        "- Are any Client, User, or Banking relationship segments under-reviewed at the selected thresholds?",
    ]
)
assert "headline accuracy" in memo
assert "precision" in memo.lower() and "recall" in memo.lower()
print(memo)

## Cross-Reference: v0.7 Interpretability And Model Risk

This notebook remains the first governance bridge at v0.1 depth. For the full model-risk treatment — per-alert explanations, feature importance and partial dependence, threshold selection across capacity and cost, false-positive concentration, the model-documentation template, the monitoring checklist, and the stakeholder governance memo — continue to the v0.7 module:

- `notebooks/07_interpretability_model_risk/alpine_crest_interpretability.ipynb`
- `notebooks/07_interpretability_model_risk/novabank_interpretability.ipynb`
- `notebooks/07_interpretability_model_risk/governance_memo.ipynb`

The `07` module cross-references back here: it deepens, rather than replaces, this first governance bridge.